# ToolCall-SFT-v1 — 150K CPU-Colab Builder

This notebook builds the complete teacher-free, single-turn ToolCall-SFT-v1 dataset. It is CPU-only, persists every stage to Google Drive, automatically resumes after a runtime restart, runs a 1,000-row gate, and verifies exactly 150,000 final records.

The target protocol matches ToolCall-200M pretraining: `call`, `ask_clarification`, or `no_call`. The uncalibrated `confidence` field is deliberately omitted.

In [ ]:
# Cell 1 — Install the small CPU-only dependency set
import subprocess, sys

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "datasets==3.6.0",
        "huggingface-hub==0.31.4",
        "sentencepiece==0.2.0",
        "tqdm==4.67.1",
    ],
    check=True,
)
print("Dependencies ready. Python:", sys.version.split()[0])

In [ ]:
# Cell 2 — Upload and locate the builder project robustly
from pathlib import Path, PurePosixPath
from google.colab import files
import shutil, zipfile

PROJECT = Path("/content/ToolCall-SFT-v1-CPU-Colab")
MARKER = ".toolcall_sft_v1_project"
EXPECTED_VERSION = "1.0.3"

def safe_extract_zip(archive: Path, destination: Path) -> None:
    with zipfile.ZipFile(archive) as zf:
        for info in zf.infolist():
            member = PurePosixPath(info.filename)
            if member.is_absolute() or ".." in member.parts:
                raise RuntimeError(f"Unsafe ZIP member: {info.filename}")
        zf.extractall(destination)

installed_version = (PROJECT / "VERSION").read_text().strip() if (PROJECT / "VERSION").is_file() else None
if not (PROJECT / MARKER).is_file() or installed_version != EXPECTED_VERSION:
    print(f"Upload ToolCall-SFT-v1-CPU-Colab-v{EXPECTED_VERSION}.zip")
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if zipfile.is_zipfile(name)]
    if len(archives) != 1:
        raise RuntimeError(f"Upload exactly one project ZIP; found {len(archives)} valid ZIPs")
    unpack = Path("/content/toolcall_sft_project_unpack")
    if unpack.exists():
        shutil.rmtree(unpack)
    unpack.mkdir(parents=True)
    safe_extract_zip(archives[0], unpack)
    candidates = [path.parent for path in unpack.rglob(MARKER)]
    if len(candidates) != 1:
        found = [str(path.relative_to(unpack)) for path in unpack.rglob("sft_builder.py")]
        raise RuntimeError(
            f"Expected one marked ToolCall SFT project, found {len(candidates)}. "
            f"Builder files found: {found}"
        )
    extracted_version = (candidates[0] / "VERSION").read_text().strip() if (candidates[0] / "VERSION").is_file() else None
    if extracted_version != EXPECTED_VERSION:
        raise RuntimeError(f"Expected project version {EXPECTED_VERSION}, found {extracted_version!r}")
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    shutil.copytree(candidates[0], PROJECT)
    shutil.rmtree(unpack)

BUILDER = PROJECT / "scripts" / "sft_builder.py"
CONFIG = PROJECT / "configs" / "sft_v1.json"
if not BUILDER.is_file() or not CONFIG.is_file():
    raise RuntimeError(f"Incomplete project extracted at {PROJECT}")
print("Project ready:", PROJECT, "version", EXPECTED_VERSION)

In [ ]:
# Cell 3 — Mount Drive and create the persistent dataset root
from google.colab import drive
drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/ToolCall-SFT-v1")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
for directory in ("state", "source", "pilot", "work", "final", "tokenizer"):
    (DATA_ROOT / directory).mkdir(parents=True, exist_ok=True)
print("Persistent root:", DATA_ROOT)

In [ ]:
# Cell 4 — Supply Hugging Face access only when the source is not cached
from getpass import getpass
import os

NORMALIZED = DATA_ROOT / "source" / "normalized_single_call_seeds.jsonl"
NORMALIZED_MARKER = DATA_ROOT / "state" / "01_normalized.json"
if NORMALIZED.is_file() and NORMALIZED_MARKER.is_file():
    print("Normalized xLAM source is already cached; no HF token is needed.")
else:
    hf_token = getpass("Paste a Hugging Face read token with xLAM access: " ).strip()
    if not hf_token:
        raise RuntimeError("A Hugging Face token is required for the first normalization run")
    os.environ["HF_TOKEN"] = hf_token
    print("Token loaded into this process only; it will not be written to Drive.")

In [ ]:
# Cell 5 — Install and validate the frozen 32K ToolCall tokenizer
import sentencepiece as spm

TOKENIZER = DATA_ROOT / "tokenizer" / "toolcall_spm_32k.model"
if not TOKENIZER.is_file():
    print("Upload toolcall_spm_32k.model")
    uploaded = files.upload()
    models = [Path(name) for name in uploaded if Path(name).suffix == ".model"]
    if len(models) != 1:
        raise RuntimeError(f"Upload exactly one SentencePiece .model file; found {len(models)}")
    shutil.copy2(models[0], TOKENIZER)

sp = spm.SentencePieceProcessor(model_file=str(TOKENIZER))
if sp.vocab_size() != 32_000:
    raise RuntimeError(f"Wrong tokenizer: expected 32,000 pieces, found {sp.vocab_size():,}")
print("Tokenizer ready:", TOKENIZER)
print("Vocabulary:", f"{sp.vocab_size():,}", "EOS ID:", sp.eos_id())

In [ ]:
# Cell 6 — Show resumable state before doing work
subprocess.run(
    [sys.executable, str(BUILDER), "status", "--root", str(DATA_ROOT), "--config", str(CONFIG)],
    check=True,
)

In [ ]:
# Cell 7 — Download, filter, normalize, and split eligible xLAM seeds
subprocess.run(
    [
        sys.executable, str(BUILDER), "normalize",
        "--root", str(DATA_ROOT),
        "--config", str(CONFIG),
        "--tokenizer", str(TOKENIZER),
    ],
    check=True,
)

In [ ]:
# Cell 8 — Build and automatically validate the 1,000-row quality gate
from collections import deque

def run_builder(stage):
    command = [
        sys.executable, "-u", str(BUILDER), stage,
        "--root", str(DATA_ROOT),
        "--config", str(CONFIG),
        "--tokenizer", str(TOKENIZER),
    ]
    print(f"Launching builder stage: {stage}")
    print("Command:", " ".join(command), flush=True)
    tail = deque(maxlen=120)
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail.append(line)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"Builder stage {stage!r} failed with return code {return_code}.\n\n"
            "Final builder output:\n" + "".join(tail)
        )

run_builder("pilot")

# Print six deterministic examples without loading the full pilot into memory.
pilot_path = DATA_ROOT / "pilot" / "pilot_1000.jsonl"
shown = set()
with pilot_path.open(encoding="utf-8") as handle:
    for line in handle:
        record = __import__("json").loads(line)
        if record["category"] in shown:
            continue
        shown.add(record["category"])
        print("\n---", record["category"], "---")
        print("USER:", record["request"][:500])
        print("TOOLS:", [tool["name"] for tool in record["tools"]])
        print("TARGET:", record["target_text"])
        if len(shown) == 6:
            break

## Full generation

The next cell generates all 150,000 accepted rows. It checkpoints each category/split file to Drive and prints progress every 500 accepted records. If Colab disconnects, reconnect, rerun Cells 1–6, and then rerun this cell; completed buckets are skipped and partial buckets resume.

In [ ]:
# Cell 9 — Generate the complete 150K candidate set
# On first v1.0.2+ run, the builder archives only the incomplete old
# clarification bucket. Completed buckets such as train/valid_call stay intact.
if "run_builder" not in globals():
    raise RuntimeError("Run Cell 8 once in this runtime to define run_builder")
run_builder("build")

In [ ]:
# Cell 10 — Export balanced splits and perform the final full verification
# If v1.0.2 already exported the 150K files, export is safe but unnecessary;
# run_builder("verify") alone completes the v1.0.3 recovery.
if "run_builder" not in globals():
    raise RuntimeError("Run Cell 8 once in this runtime to define run_builder")
run_builder("export")
run_builder("verify")

In [ ]:
# Cell 11 — Inspect final statistics and file sizes
import json

FINAL = DATA_ROOT / "final"
statistics_value = json.loads((FINAL / "statistics.json").read_text())
verification = json.loads((FINAL / "verification_report.json").read_text())
print(json.dumps(statistics_value, indent=2))
print("\nVerification:", json.dumps(verification, indent=2))
print("\nFinal files:")
for path in sorted(FINAL.iterdir()):
    if path.is_file():
        print(f"  {path.name:28s} {path.stat().st_size / 1024**2:9.2f} MiB")

In [ ]:
# Cell 12 — Optional: create a Drive archive for later Kaggle upload
# The verified final directory is already safe in Drive. Set this to True only
# when you want a single archive; compression can take additional CPU time.
CREATE_ARCHIVE = False

if CREATE_ARCHIVE:
    import tarfile
    archive = DATA_ROOT.parent / "ToolCall-SFT-v1-150k.tar.gz"
    temporary = archive.with_suffix(archive.suffix + ".partial")
    with tarfile.open(temporary, "w:gz") as tf:
        tf.add(FINAL, arcname="ToolCall-SFT-v1/final")
        tf.add(TOKENIZER, arcname="ToolCall-SFT-v1/tokenizer/toolcall_spm_32k.model")
    temporary.replace(archive)
    print("Archive ready:", archive, f"({archive.stat().st_size / 1024**3:.2f} GiB)")
else:
    print("Archive skipped. Verified data remains at:", FINAL)